# Question 2: Topic modelling of South African news with LDA

**Analytics flow**
1. **2.1** Load the news articles into a data frame, clean the text, build a document-term matrix.
2. **2.2** Fit a Latent Dirichlet Allocation model (3 topics) and inspect the top words (beta) and document-topic mix (gamma).
3. **2.3** Visualise the topics as faceted bar charts and word clouds.

> **Dataset.** The notebook expects `data/q2_news/news.zip` (a folder of `.txt` articles, `sa_news/`). It was not part of the folder that was uploaded, so add it before running. See `data/q2_news/README.md`.

In [ ]:
library(tidyverse)
library(tm)
library(topicmodels)
library(tidytext)
library(ggplot2)
library(wordcloud)
library(RColorBrewer)

## 2.1 Data frame creation and preprocessing

In [ ]:
news_zip    <- "../data/q2_news/news.zip"
extract_dir <- "../data/q2_news/extracted_news"
if (!file.exists(news_zip)) stop("Add news.zip to data/q2_news/ first (see data/q2_news/README.md).")

unzip(news_zip, exdir = extract_dir)
file_paths <- list.files(extract_dir, pattern = "\\.txt$", full.names = TRUE, recursive = TRUE)

news_df <- data.frame(
  file_name = basename(file_paths),
  content = sapply(file_paths, function(path) {
    readLines(path, warn = FALSE, encoding = "UTF-8") %>% paste(collapse = " ")
  }, USE.NAMES = FALSE),
  stringsAsFactors = FALSE
)
print("--- Data Frame Preview ---")
print(head(news_df))

**Recorded output** (screenshot from the project documentation): news data frame preview

![news data frame preview](../docs/images/q2_1_dataframe.png)

Clean the text: lower-case, strip punctuation and numbers, remove English stop words and extra white space.

In [ ]:
corpus <- VCorpus(VectorSource(news_df$content))
corpus <- tm_map(corpus, content_transformer(tolower))
corpus <- tm_map(corpus, removePunctuation)
corpus <- tm_map(corpus, removeNumbers)
corpus <- tm_map(corpus, removeWords, stopwords("english"))
corpus <- tm_map(corpus, stripWhitespace)

dtm <- DocumentTermMatrix(corpus)

# LDA cannot take empty documents, and very rare terms add noise
row_totals <- apply(dtm, 1, sum)
dtm <- dtm[row_totals > 0, ]
dtm <- removeSparseTerms(dtm, 0.99)
dtm

**Recorded output** (screenshot from the project documentation): document-term matrix

![document-term matrix](../docs/images/q2_1_dtm.png)

## 2.2 Latent Dirichlet Allocation (LDA)

In [ ]:
n_topics <- 3
k_words  <- 5

set.seed(42)
lda_model <- LDA(dtm, k = n_topics, control = list(seed = 42))

topic_word_probs <- tidy(lda_model, matrix = "beta")

top_terms <- topic_word_probs %>%
  group_by(topic) %>%
  slice_max(beta, n = k_words) %>%
  ungroup() %>%
  arrange(topic, -beta)

print("Top words and word distribution (beta) per topic")
print(top_terms)

**Recorded output** (screenshot from the project documentation): top terms per topic

![top terms per topic](../docs/images/q2_2_top_terms.png)

In [ ]:
doc_topics_probs <- tidy(lda_model, matrix = "gamma")
print("Document-topic distribution (gamma) preview")
print(head(doc_topics_probs))

**Recorded output** (screenshot from the project documentation): document-topic probabilities

![document-topic probabilities](../docs/images/q2_2_gamma.png)

## 2.3 Visualisation

In [ ]:
vis_plot_1 <- top_terms %>%
  mutate(term = reorder_within(term, beta, topic)) %>%
  ggplot(aes(x = beta, y = term, fill = factor(topic))) +
  geom_col(show.legend = FALSE) +
  facet_wrap(~ topic, scales = "free") +
  scale_y_reordered() +
  labs(
    title = "Top Keywords in Each Topic (LDA Model)",
    subtitle = "Faceted by Topic with Word Distribution Probabilities (Beta)",
    x = "Word Distribution Probability (Beta)",
    y = "Terms"
  ) +
  theme_minimal()

print(vis_plot_1)

**Recorded output** (screenshot from the project documentation): top keywords per topic

![top keywords per topic](../docs/images/q2_3_topic_barplot.png)

In [ ]:
par(mfrow = c(1, n_topics))
for (i in 1:n_topics) {
  topic_data <- topic_word_probs %>% filter(topic == i)
  wordcloud(
    words = topic_data$term,
    freq = topic_data$beta,
    max.words = 20,
    random.order = FALSE,
    colors = brewer.pal(8, "Dark2")
  )
  title(main = paste("Topic", i))
}
par(mfrow = c(1, 1))

**Recorded output** (screenshot from the project documentation): word clouds per topic

![word clouds per topic](../docs/images/q2_3_wordclouds.png)

## Reading the result

The recorded run shows three loosely separated themes: public administration and unemployment (topic 1), government departments and Eskom (topic 2), and invoices, coalition and national government (topic 3). The word *said* tops several topics because it is a news-reporting verb rather than a subject word; adding it to the stop-word list is a natural next refinement.

*Changes from `src/q2_topic_modelling_lda.R`:* `stringAsFactors` (typo, which created an extra column) is now `stringsAsFactors`, and files are found recursively so the zip's folder name does not matter.